In [1]:
%matplotlib widget
import torch

from thesis_code.datasets import get_SHD_dataloader
from thesis_code.networks import train, test, models
from thesis_code.plotting import plot_performance, plot_gradients, InteractiveSpikePlot

In [2]:
torch.manual_seed(42)
dtype = torch.float
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

Timesteps are in milliseconds

In [3]:
dt = 0.001

In [4]:
SHD_trainloader, input_dim, classes = get_SHD_dataloader(
    batch_size=128,
    time_window=1000,
    train=True,
    shuffle=True,
    drop_last=True,
    re_download=False,
)

SHD_testloader, _, _ = get_SHD_dataloader(
    batch_size=128,
    time_window=1000,
    train=False,
    shuffle=True,
    drop_last=True,
    re_download=False,
)

In [5]:
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import numpy as np
import torch.nn.functional as F
from snntorch import surrogate
from typing import Callable

from thesis_code.training import get_compute_loss_reg_fn
from thesis_code.custom_objects.building_blocks import sample_heterogeneous_decays_normal, tau_to_beta

def compute_loss(mem_rec: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    probs = F.softmax(mem_rec, dim=-1)
    log_probs = torch.log(probs.mean(dim=0) + 1e-8) # to avoid log(0)
    return F.nll_loss(log_probs, targets)

def measure_accuracy(mem_rec, targets):
    _, idx = mem_rec.sum(dim=0).max(1)
    return np.mean((targets == idx).detach().cpu().numpy())

def test(net: nn.Module, device: torch.device, testloader: DataLoader, max_iters: int = None) -> float:
    with torch.no_grad():
        acc = 0
        net.eval()

        for i, (data, targets) in enumerate(testloader):
            data, targets = data.to(device), targets.to(device).long()
            data = data.squeeze(2)
            
            spk_rec, mem_rec, hidden_spks = net(data)
            acc += measure_accuracy(mem_rec, targets)
            if max_iters is not None and i == max_iters:
                break

    return acc / len(testloader)

def record_grads(net: nn.Module) -> tuple[list, list]:
    grads = {name: p.grad.detach().clone()
        for name, p in net.named_parameters() if p.grad is not None}
    if len(grads) > 0:
        flat_grads = torch.cat([t.flatten() for t in grads.values()])
        return flat_grads.max(dim=0).values.cpu().numpy(), flat_grads.mean(dim=0).cpu().numpy()
    return 0, 0

def train(net: nn.Module, device: torch.device, trainloader: DataLoader, loss_fn: Callable, lr: float = 1e-3, n_epochs: int = 1, max_iters: int = None) -> tuple[nn.Module, list, list]:
    optimizer = torch.optim.Adam(net.parameters(), lr=lr, eps=1e-6, betas=(0.9, 0.99))

    loss_hist, acc_hist, spike_hist = [], [], []
    net.train()
    max_grad_rec = []
    avg_grad_rec = []

    for epoch in range(n_epochs):
        progress_bar = tqdm(enumerate(trainloader), total=len(trainloader), desc=f"Epoch {epoch}")
        for i, (data, targets) in progress_bar:
            data, targets = data.to(device), targets.to(device).long()
            data = data.squeeze(2)

            spk_outs, mem_outs, hidden_spks = net(data)
            loss_val = loss_fn(mem_outs, targets, hidden_spks)

            optimizer.zero_grad()
            loss_val.backward()

            # torch.nn.utils.clip_grad_value_(net.parameters(), clip_value=1e6)

            max_grad, avg_grad = record_grads(net)
            max_grad_rec.append(max_grad)
            avg_grad_rec.append(avg_grad)

            optimizer.step()

            loss_hist.append(loss_val.item())

            acc = measure_accuracy(mem_outs, targets)
            acc_hist.append(acc)

            progress_bar.set_postfix(loss=f"{loss_val.item():.2f}", acc=f"{acc * 100:.2f}%")

            if i % 25 == 0:
                tqdm.write(f"Epoch {epoch}, Iteration {i} — Loss: {loss_val.item():.2f}, Acc: {acc*100:.2f}%")

            if max_iters is not None and i == max_iters:
                break

    return net, loss_hist, acc_hist, spike_hist, max_grad_rec, avg_grad_rec

In [6]:
from thesis_code.connectivity import sample_ee_pv_som_neurons, build_rec_matrices

ns_hidden=[256, 256]
neurons = sample_ee_pv_som_neurons(ns_hidden, percent_pv=25.0, percent_som=25.0)
rec_matrices_pv_som_ee = build_rec_matrices(neurons)

In [7]:
layers = [input_dim] + ns_hidden

net_pv_som_ee = models.StandardSRNN(
    forward_matrices=[torch.ones(n_in, n_out) for n_in, n_out in zip(layers[:-1], layers[1:])],
    recurrent_matrices=rec_matrices_pv_som_ee,
    betas=[sample_heterogeneous_decays_normal(dt, n, tau_mean=10 * dt, tau_std=1 * dt, device=device) for n in ns_hidden],
    beta_out=tau_to_beta(dt, 20 * dt),
    n_classes=len(classes),
    spike_grad=surrogate.fast_sigmoid(slope=25),
).to(device)

compute_loss_reg = get_compute_loss_reg_fn(
    lam_lower=100.0,
    v_lower=1e-2,               # min n spikes
    lam_uppers=[0.5, 0.5],
    v_uppers=[50, 50],          # max n spikes per layer
    L=2,
)

accuracy = test(
    net=net_pv_som_ee,
    device=device,
    testloader=SHD_testloader,
    max_iters=None,
)
print(f"Test Accuracy before training is: {accuracy}")

net_pv_som_ee, loss_hist, acc_hist, spike_hist, max_grad_rec, avg_grad_rec = train(
    net=net_pv_som_ee,
    device=device,
    trainloader=SHD_trainloader,
    loss_fn=lambda mem_outs, targets, hidden_spks: compute_loss(mem_outs, targets) + compute_loss_reg(hidden_spks),
    lr=1e-3,
    n_epochs=50,
    max_iters=None,
)
plot_performance(loss_hist, acc_hist)
# plot_gradients(max_grad_rec, avg_grad_rec)

accuracy = test(
    net=net_pv_som_ee,
    device=device,
    testloader=SHD_testloader,
    max_iters=None,
)
print(f"Test Accuracy after training is: {accuracy}")

Test Accuracy before training is: 0.051011029411764705


Epoch 0:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 0, Iteration 0 — Loss: 3.58, Acc: 1.56%
Epoch 0, Iteration 25 — Loss: 2.86, Acc: 11.72%
Epoch 0, Iteration 50 — Loss: 2.11, Acc: 32.03%


Epoch 1:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 1, Iteration 0 — Loss: 1.97, Acc: 39.06%
Epoch 1, Iteration 25 — Loss: 1.60, Acc: 56.25%
Epoch 1, Iteration 50 — Loss: 1.54, Acc: 45.31%


...

Epoch 26:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 26, Iteration 0 — Loss: 0.40, Acc: 92.97%
Epoch 26, Iteration 25 — Loss: 0.33, Acc: 95.31%
Epoch 26, Iteration 50 — Loss: 0.47, Acc: 91.41%


Epoch 27:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 27, Iteration 0 — Loss: 0.40, Acc: 91.41%


RuntimeError: [enforce fail at alloc_cpu.cpp:117] data. DefaultCPUAllocator: not enough memory: you tried to allocate 14942208 bytes.